<a href="https://colab.research.google.com/github/arthawebid/Deep_learning_project01/blob/main/Project_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hybrid Automated Essay Scoring menggunakan Deep Learning (BiLSTM) dan Cosine Similarity

Siapkan Modul yang digunakan

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model

# Upload Dataset

In [ ]:
from google.colab import files
import io

uploaded = files.upload()

filename = list(uploaded.keys())[0]

data = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Dataset shape:", data.shape)

data.head()

# Ambil Teks dan Label

## Tujuan
Memisahkan fitur dan label.

In [ ]:
texts = data["essay_text"].astype(str)
references = data["reference_answer"].astype(str)
scores = data["final_score"].values

# TF-IDF Vectorization
Mengubah teks menjadi angka agar bisa diproses secara matematis.

In [ ]:
# TF-IDF untuk semua dokumen.
vectorizer = TfidfVectorizer()

tfidf = vectorizer.fit_transform(
    references.tolist() + texts.tolist()
)

# Cosine Similarity
Mengukur kemiripan antara esai mahasiswa dan jawaban referensi.

Rumus cosine similarity:

$\cos(\theta)=\frac{A\cdot B}{||A||,||B||}$

In [ ]:
# Hitung similarity setiap pasangan reference vs essay.
similarities = cosine_similarity(
    tfidf[:len(data)],
    tfidf[len(data):]
).diagonal()

data["similarity"] = similarities

# Cek dataset:
data[["essay_text","reference_answer","similarity"]].head()

# Tokenization, Padding dan Dataset Split

## Tokenization
Mengubah teks menjadi angka indeks kata. Ini diperlukan karena Deep Learning hanya memproses angka.

sebagai contoh:  
text = "Teknologi digital membantu pendidikan"    
Token = [[3], [4], [7], [2], [8], [9], [8], [10], [1], [], [5], [1], [10], [1], [3], [6], [9], [], [11], [4], [11], [12], [6], [2], [3], [13], [], [14], [4], [2], [5], [1], [5], [1], [7], [6], [2]]


## Padding
Menyamakan panjang semua esai.

## Dataset Split
Mempersiapkan fitur model.


In [ ]:
# Tokenisasi
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

# Padding
max_length = 200
padded = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post"
)
print("Text tensor shape:", padded.shape)

# Dataset Split
# Input model terdiri dari 2 fitur: teks dan similarity
X_text = padded
X_sim = data["similarity"].values
y = scores

# Split dataset:
X_text_train, X_text_test, X_sim_train, X_sim_test, y_train, y_test = train_test_split(
    X_text,
    X_sim,
    y,
    test_size=0.2,
    random_state=42
)

Text tensor shape: (5000, 200)


# Model Deep Learning
## Input Layer
Menerima teks sepanjang 200 token.

## Embedding Layer
Mengubah kata menjadi vektor semantik.

## LSTM
Digunakan untuk memahami urutan kata dalam kalimat.

## Dense Layer
Layer neural network untuk menggabungkan fitur yang dipelajari.

## Input Similarity
Memasukkan nilai cosine similarity sebagai fitur tambahan.

## Menggabungkan Fitur
menggabungkan teks dan similarity, Sehingga model memiliki informasi bahasa + relevansi jawaban.

## Output Layer
Menghasilkan prediksi skor esai.


In [ ]:
# Input teks
text_input = Input(shape=(max_length,))

# Embedding layer
embedding = Embedding(10000,128)(text_input)

# BiLSTM layer
lstm = Bidirectional(LSTM(64))(embedding)

# Dense layer
dense_text = Dense(32, activation="relu")(lstm)

# Input similarity
sim_input = Input(shape=(1,))

# Gabungkan fitur
concat = Concatenate()([dense_text, sim_input])

# Dense layer
dense = Dense(16, activation="relu")(concat)

# Output score
output = Dense(1)(dense)

# Buat model
model = Model(
    inputs=[text_input, sim_input],
    outputs=output
)

# Compile model
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Cek struktur mode
model.summary()

# Training Model

Melatih model agar memahami hubungan:  
```
teks esai + similarity → skor esai
```
teks esai + similarity → skor esai
```
10 epoch
```


In [ ]:
history = model.fit(
    [X_text_train, X_sim_train],
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

# Evaluasi Model
## RMSE
Mengukur rata-rata error prediksi model. Semakin kecil semakin baik.

## MAE
Mengukur rata-rata selisih skor prediksi dengan skor sebenarnya.  
MAE = 0.4  
rata-rata kesalahan 0.4 poin



In [ ]:
pred = model.predict([X_text_test, X_sim_test]).flatten()

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)

# Grafik Training

In [ ]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(["Train","Validation"])

plt.show()

# Prediksi Esai Baru

Cosine similarity berada pada rentang:
```
0  → tidak mirip sama sekali
1  → identik
```

Kategori interpretasi yang sering digunakan:
| Range Similarity | Kategori       | Interpretasi                         |
| ---------------- | -------------- | ------------------------------------ |
| 0.00 – 0.30      | Tidak relevan  | Jawaban tidak sesuai dengan topik    |
| 0.30 – 0.50      | Kurang relevan | Jawaban sedikit terkait              |
| 0.50 – 0.70      | Cukup relevan  | Jawaban cukup sesuai                 |
| 0.70 – 0.85      | Relevan        | Jawaban sesuai dengan referensi      |
| 0.85 – 1.00      | Sangat relevan | Jawaban hampir sama dengan referensi |

Contoh interpretasi:
| Similarity | Interpretasi   |
| ---------- | -------------- |
| 0.21       | Tidak relevan  |
| 0.44       | Kurang relevan |
| 0.63       | Cukup relevan  |
| 0.78       | Relevan        |
| 0.92       | Sangat relevan |



In [ ]:
# ===============================
# Essay baru
# ===============================

essay = """
Teknologi digital memberikan banyak perubahan dalam sistem pendidikan
dan memungkinkan mahasiswa belajar lebih fleksibel menggunakan internet.
"""


# ===============================
# Preprocessing teks
# ===============================

seq = tokenizer.texts_to_sequences([essay])

pad = pad_sequences(
    seq,
    maxlen=max_length,
    padding="post"
)

pad = np.asarray(pad).astype("int32")


# ===============================
# Hitung cosine similarity
# ===============================

reference = data["reference_answer"].iloc[0]

vectorizer = TfidfVectorizer()

tfidf = vectorizer.fit_transform([reference, essay])

similarity = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]


# ===============================
# Bentuk input similarity
# ===============================

sim_input = np.asarray([[similarity]]).astype("float32")


# ===============================
# Cek shape
# ===============================

print("Text input shape:", pad.shape)
print("Similarity input shape:", sim_input.shape)


# ===============================
# Prediksi
# ===============================

score = model([pad, sim_input], training=False).numpy()

print("Similarity:", similarity)
print("Predicted Score:", float(score[0][0]))

Text input shape: (1, 200)
Similarity input shape: (1, 1)
Similarity: 0.24246388420852777
Predicted Score: 3.0812997817993164
